
## Bronze Layer Ingestion (Demo)

This demo notebook shows how to ingest JSON files from **S3 or Unity Catalog Volumes** using **Databricks Auto Loader**.

### Key Features
- Uses **Auto Loader (cloudFiles)** for incremental file ingestion
- Automatically manages **schema inference & evolution**
- Stores:
  - **Checkpoint** in Volume (for fault tolerance)
  - **Schema** in Volume (for schema tracking)
- Writes data into a **Delta table (Bronze layer)**
- Adds **metadata columns** (ingestion time, file details, record hash)

### Notes
- Designed to be triggered via a **Databricks Job**
- Uses `availableNow` trigger for batch-style streaming execution.

In [0]:
from pyspark.sql.functions import *

In [0]:
# Jobs params
# config_table = "dev_bronze.meta.config_table"
# source_type = "streaming"
config_table = dbutils.widgets.get("config_table")
source_type = dbutils.widgets.get("source_type")

In [0]:
# Load configuration for the given source_type, log total vs active tables,
# and fail the job if no active tables are available for processing

config_df = spark.read.table(config_table).filter(col("source_type") == source_type)
total_counts = config_df.count()
configs = config_df.filter(col("load_in_progress").isNull() & (col("is_active") == True)).collect()
total_active_counts = len(configs)

print(f"Total tables = {total_counts}, Active tables = {total_active_counts} for source_type = '{source_type.upper()}'\n")

if total_active_counts == 0:
    raise ValueError(f"No active tables found for source_type='{source_type}'. Failing the job.\n")

In [0]:
def bronze_layer_ingestion(
    source_file_path, 
    source_file_format, 
    target_table, 
    checkpoint_path, 
    schema_path
):
    # Configure Auto Loader to ingest JSON data to a Delta table
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", source_file_format)
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(source_file_path)
    )
    # add a metadata column 
    df = (
        df.withColumn(
            "meta_data",
            struct(
                current_timestamp().alias("bronze_ingestion_ts"),
                col("_metadata.file_path").alias("source_file"),
                col("_metadata.file_size").alias("file_size"),
                col("_metadata.file_modification_time").alias("file_modification_time"),
                sha2(concat_ws("||", *df.columns), 256).alias("record_hash")  
            )
        ).withColumn("bronze_load_date", to_date(current_timestamp()))
    )
    
    # wrtie stream to table
    return (
        df.writeStream
        .option("checkpointLocation", checkpoint_path)
        .partitionBy("bronze_load_date")
        .trigger(availableNow=True)
        .toTable(target_table)
    )


In [0]:
for item in configs:
    id  = item["id"]
    source = item["source"]
    source_type = item["source_type"]
    target_table = item["bronze_cat_schema_table"]
    catalog, schema, table_name = target_table.split(".")

    # update config table
    update_config_table = f"UPDATE {config_table} SET load_in_progress = 'B' WHERE id = {id}"
    spark.sql(update_config_table)

    if source_type.lower() == "streaming":
        source_file_format = item["source_format"]
        base_path = f"/Volumes/{catalog}/meta/pipeline_metadata/autoloader"
        checkpoint_path = f"{base_path}/checkpoints/{schema}/{table_name}"
        schema_path = f"{base_path}/schema/{schema}/{table_name}"

        # Truncate table and remove any chekpoints from above path (this is for demo purpose only)
        if spark.catalog.tableExists(target_table): 
            spark.sql(f"TRUNCATE TABLE {target_table}")
        dbutils.fs.rm(checkpoint_path, True)
        dbutils.fs.rm(schema_path, True)
        
        streaming = bronze_layer_ingestion(
            source_file_path=source,
            source_file_format=source_file_format,
            target_table=target_table, 
            checkpoint_path=checkpoint_path, 
            schema_path=schema_path
        )

        # Wait for ingestion to complete
        streaming.awaitTermination()
        ingested_count = spark.table(target_table).count()

        print(f"{target_table}, ingested records: {ingested_count}")
    
    # reset config table
    reset_config_table = f"""
        UPDATE {config_table} 
        SET load_in_progress = NULL, bronze_layer_update_ts = CURRENT_TIMESTAMP()
        WHERE id = {id}
    """
    spark.sql(reset_config_table)
    
    # TODO: Ingestion for other source types

